# HfCacheManager Tutorial: Intelligent Cache Management for HuggingFace Datasets

The `HfCacheManager` class provides sophisticated cache management capabilities for HuggingFace genomics datasets. It extends DataCard functionality with intelligent caching strategies and automated cache cleanup tools.

This tutorial covers:
- Setting up HfCacheManager for cache management
- Understanding the 3-case metadata caching strategy
- Automated cache cleanup by age, size, and revision
- Cache monitoring and diagnostics
- Best practices for efficient cache management
- Integration with data loading workflows

**Prerequisites**: Basic familiarity with DataCard (see datacard_tutorial.ipynb) and HuggingFace datasets.

## 1. Setting Up HfCacheManager

The HfCacheManager extends DataCard with cache management capabilities. Unlike DataCard which focuses on dataset exploration, HfCacheManager adds intelligent caching and cleanup features.

In [1]:
import duckdb
import logging
from labretriever.hf_cache_manager import HfCacheManager
from huggingface_hub import scan_cache_dir

# Set up logging to see cache management activities
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Create DuckDB connection for metadata caching
conn = duckdb.connect(':memory:')

# Initialize HfCacheManager
cache_manager = HfCacheManager(
    repo_id='BrentLab/mahendrawada_2025',
    duckdb_conn=conn,
    logger=logger
)

print(f"HfCacheManager initialized for: {cache_manager.repo_id}")
print(f"DuckDB connection: {'Active' if conn else 'None'}")
print(f"Logger configured: {'Yes' if logger else 'No'}")

# Show current cache status -- NOTE: this is from huggingface_hub,
# not from HfCacheManager
cache_info = scan_cache_dir()
print(f"Current HF cache size: {cache_info.size_on_disk_str}")
print(f"Cached repositories: {len(cache_info.repos)}")

/home/chase/code/tfbp/labretriever/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


HfCacheManager initialized for: BrentLab/mahendrawada_2025
DuckDB connection: Active
Logger configured: Yes
Current HF cache size: 5.5G
Cached repositories: 11


In [2]:
cache_info

HFCacheInfo(size_on_disk=5525278711, repos=frozenset({CachedRepoInfo(repo_id='BrentLab/yeast_comparative_analysis', repo_type='dataset', repo_path=PosixPath('/home/chase/.cache/huggingface/hub/datasets--BrentLab--yeast_comparative_analysis'), size_on_disk=166072, nb_files=7, revisions=frozenset({CachedRevisionInfo(commit_hash='ac03d065bb493bc9dd7a77460fdaf6f954968b0b', snapshot_path=PosixPath('/home/chase/.cache/huggingface/hub/datasets--BrentLab--yeast_comparative_analysis/snapshots/ac03d065bb493bc9dd7a77460fdaf6f954968b0b'), size_on_disk=166072, files=frozenset({CachedFileInfo(file_name='part-0.parquet', file_path=PosixPath('/home/chase/.cache/huggingface/hub/datasets--BrentLab--yeast_comparative_analysis/snapshots/ac03d065bb493bc9dd7a77460fdaf6f954968b0b/dto/binding_repo_dataset=harbison_2004-harbison_2004/perturbation_repo_dataset=hu_2007_reimand_2010-hu_2007_reimand_2010/part-0.parquet'), blob_path=PosixPath('/home/chase/.cache/huggingface/hub/datasets--BrentLab--yeast_comparative

## 2. Understanding the 3-Case Metadata Caching Strategy

HfCacheManager implements an intelligent 3-case strategy for metadata access that minimizes downloads and maximizes performance:

1. **DuckDB Check**: First check if metadata already exists in the DuckDB database
2. **Cache Load**: If not in DuckDB, try to load from local HuggingFace cache  
3. **Download**: If not cached, download from HuggingFace Hub

This strategy is implemented in the internal `_get_metadata_for_config()` method and automatically used when loading data with HfQueryAPI.

### Demonstrating the 3-Case Strategy

Let's see how the caching strategy works by examining what metadata tables would be created and checking the DuckDB state.

In [3]:
# Check current DuckDB state (Case 1 check)
tables = conn.execute(
    "SELECT table_name FROM information_schema.tables WHERE table_name LIKE 'metadata_%'"
).fetchall()

print("DuckDB Metadata Tables (Case 1):")
print("=" * 35)
if tables:
    for table in tables:
        count = conn.execute(f"SELECT COUNT(*) FROM {table[0]}").fetchone()[0]
        print(f"  • {table[0]}: {count} rows")
else:
    print("  No metadata tables found in DuckDB")
    print("  → Would proceed to Case 2 (check HF cache) or Case 3 (download)")

print(f"\nThe 3-case strategy ensures:")
print("• Fast access: DuckDB queries are nearly instantaneous")
print("• Minimal downloads: Reuse locally cached files when possible")  
print("• Automatic fallback: Download only when necessary")
print("• Transparent operation: Works automatically with HfQueryAPI")

DuckDB Metadata Tables (Case 1):
  No metadata tables found in DuckDB
  → Would proceed to Case 2 (check HF cache) or Case 3 (download)

The 3-case strategy ensures:
• Fast access: DuckDB queries are nearly instantaneous
• Minimal downloads: Reuse locally cached files when possible
• Automatic fallback: Download only when necessary
• Transparent operation: Works automatically with HfQueryAPI


### Checking HuggingFace Cache Status (Case 2)

The second case checks if files are already cached locally by HuggingFace. Let's examine the cache state for our target repository.

In [4]:
# Check if target repository is in HuggingFace cache
cache_info = scan_cache_dir()
target_repo = None

for repo in cache_info.repos:
    if repo.repo_id == cache_manager.repo_id:
        target_repo = repo
        break

print("HuggingFace Cache Status (Case 2):")
print("=" * 40)

if target_repo:
    print(f"✓ Repository {cache_manager.repo_id} found in cache")
    print(f"  Size: {target_repo.size_on_disk_str}")
    print(f"  Revisions: {len(target_repo.revisions)}")
    print(f"  Files: {target_repo.nb_files}")
    
    # Show latest revision info
    if target_repo.revisions:
        latest_rev = max(target_repo.revisions, key=lambda r: r.last_modified)
        print(f"  Latest revision: {latest_rev.commit_hash[:8]}")
        print(f"  Last accessed: {latest_rev.last_modified}")
        
    print("\n  → Case 2 would succeed: Load from local cache")
else:
    print(f"✗ Repository {cache_manager.repo_id} not found in cache")
    print("  → Would proceed to Case 3: Download from HuggingFace Hub")

print(f"\nCache efficiency: Using local files avoids re-downloading {target_repo.size_on_disk_str if target_repo else 'unknown size'}")

HuggingFace Cache Status (Case 2):
✓ Repository BrentLab/mahendrawada_2025 found in cache
  Size: 94.3M
  Revisions: 4
  Files: 8
  Latest revision: af5ac9dc
  Last accessed: 1763578870.280984

  → Case 2 would succeed: Load from local cache

Cache efficiency: Using local files avoids re-downloading 94.3M


## 3. Cache Management and Cleanup

HfCacheManager's primary value is in providing sophisticated cache management. Let's explore the different cleanup strategies available.

### Cache Overview and Current Status

Before cleaning, let's understand what we're working with.

In [5]:
# Get comprehensive cache overview
cache_info = scan_cache_dir()

print("Current HuggingFace Cache Overview:")
print("=" * 40)
print(f"Total cache size: {cache_info.size_on_disk_str}")
print(f"Number of repositories: {len(cache_info.repos)}")

# Analyze cache by repository size
repo_sizes = []
for repo in cache_info.repos:
    repo_sizes.append((repo.repo_id, repo.size_on_disk, repo.size_on_disk_str, len(repo.revisions)))

# Sort by size (largest first)
repo_sizes.sort(key=lambda x: x[1], reverse=True)

print(f"\nLargest repositories (top 5):")
for repo_id, size_bytes, size_str, revisions in repo_sizes[:5]:
    print(f"  • {repo_id}: {size_str} ({revisions} revisions)")

if len(repo_sizes) > 5:
    print(f"  ... and {len(repo_sizes) - 5} more repositories")

# Calculate total revisions
total_revisions = sum(len(repo.revisions) for repo in cache_info.repos)
print(f"\nTotal revisions across all repos: {total_revisions}")

# Show age distribution
from datetime import datetime
now = datetime.now().timestamp()
old_revisions = 0
for repo in cache_info.repos:
    for rev in repo.revisions:
        age_days = (now - rev.last_modified) / (24 * 3600)
        if age_days > 30:
            old_revisions += 1

print(f"Revisions older than 30 days: {old_revisions}")
print(f"Recent revisions (≤30 days): {total_revisions - old_revisions}")

Current HuggingFace Cache Overview:
Total cache size: 5.5G
Number of repositories: 11

Largest repositories (top 5):
  • BrentLab/barkai_compendium: 3.6G (1 revisions)
  • BrentLab/hackett_2020: 817.0M (9 revisions)
  • BrentLab/kemmeren_2014: 646.2M (3 revisions)
  • BrentLab/rossi_2021: 213.7M (9 revisions)
  • BrentLab/mahendrawada_2025: 94.3M (4 revisions)
  ... and 6 more repositories

Total revisions across all repos: 50
Revisions older than 30 days: 41
Recent revisions (≤30 days): 9


### Querying Loaded Metadata

Once metadata is loaded into DuckDB, we can query it using SQL.

### Internal Cache Management Methods

HfCacheManager provides several internal methods that work behind the scenes. Let's explore what these methods do and how they integrate with the caching strategy.

## 4. Working with Specific Metadata Configurations

You can also retrieve metadata for specific configurations rather than all at once.

In [6]:
# Demonstrate understanding of internal cache methods
print("HfCacheManager Internal Methods:")
print("=" * 35)

print("\n1. _get_metadata_for_config(config)")
print("   → Implements the 3-case strategy for a specific configuration")
print("   → Returns detailed result with strategy used and success status")

print("\n2. _check_metadata_exists_in_duckdb(table_name)")
print("   → Case 1: Checks if metadata table already exists in DuckDB")
print("   → Fast check using information_schema.tables")

print("\n3. _load_metadata_from_cache(config, table_name)")
print("   → Case 2: Attempts to load from local HuggingFace cache")
print("   → Uses try_to_load_from_cache() to find cached files")

print("\n4. _download_and_load_metadata(config, table_name)")
print("   → Case 3: Downloads from HuggingFace Hub if not cached")
print("   → Uses snapshot_download() for efficient file retrieval")

print("\n5. _create_duckdb_table_from_files(file_paths, table_name)")
print("   → Creates DuckDB views from parquet files")
print("   → Handles both single files and multiple files efficiently")

print("\n6. _extract_embedded_metadata_field(data_table, field, metadata_table)")
print("   → Extracts metadata fields from data tables")
print("   → Creates separate queryable metadata views")

print("\nThese methods work together to provide:")
print("• Transparent caching that 'just works'")
print("• Minimal network usage through intelligent fallbacks")
print("• Fast metadata access via DuckDB views")
print("• Automatic handling of different file structures")

HfCacheManager Internal Methods:

1. _get_metadata_for_config(config)
   → Implements the 3-case strategy for a specific configuration
   → Returns detailed result with strategy used and success status

2. _check_metadata_exists_in_duckdb(table_name)
   → Case 1: Checks if metadata table already exists in DuckDB
   → Fast check using information_schema.tables

3. _load_metadata_from_cache(config, table_name)
   → Case 2: Attempts to load from local HuggingFace cache
   → Uses try_to_load_from_cache() to find cached files

4. _download_and_load_metadata(config, table_name)
   → Case 3: Downloads from HuggingFace Hub if not cached
   → Uses snapshot_download() for efficient file retrieval

5. _create_duckdb_table_from_files(file_paths, table_name)
   → Creates DuckDB views from parquet files
   → Handles both single files and multiple files efficiently

6. _extract_embedded_metadata_field(data_table, field, metadata_table)
   → Extracts metadata fields from data tables
   → Creates separ

## 5. Extracting Embedded Metadata

Some datasets have metadata embedded within their data files. The HfCacheManager can extract this embedded metadata into separate, queryable tables.

## 4. Embedded Metadata Extraction

One unique feature of HfCacheManager is the ability to extract embedded metadata fields from data tables into separate, queryable metadata tables.

# Demonstrate embedded metadata extraction concept
print("Embedded Metadata Extraction:")
print("=" * 35)

print("\nScenario: You have a data table with embedded metadata fields")
print("Example: genomics data with 'experimental_condition' field")

# Create sample data to demonstrate the concept
conn.execute("""
    CREATE TABLE sample_genomics_data AS 
    SELECT 
        'gene_' || (row_number() OVER()) as gene_id,
        random() * 1000 as expression_value,
        CASE 
            WHEN (row_number() OVER()) % 4 = 0 THEN 'control'
            WHEN (row_number() OVER()) % 4 = 1 THEN 'treatment_A'
            WHEN (row_number() OVER()) % 4 = 2 THEN 'treatment_B'
            ELSE 'stress_condition'
        END as experimental_condition,
        CASE 
            WHEN (row_number() OVER()) % 3 = 0 THEN 'timepoint_0h'
            WHEN (row_number() OVER()) % 3 = 1 THEN 'timepoint_6h'
            ELSE 'timepoint_24h'
        END as timepoint
    FROM range(100)
""")

print("✓ Created sample genomics data with embedded metadata fields")

# Show the data structure
sample_data = conn.execute(
    "SELECT * FROM sample_genomics_data LIMIT 5"
).fetchall()

print(f"\nSample data structure:")
print("gene_id | expression_value | experimental_condition | timepoint")
print("-" * 65)
for row in sample_data:
    print(f"{row[0]:8} | {row[1]:15.1f} | {row[2]:20} | {row[3]}")

print(f"\nEmbedded metadata fields identified:")
print("• experimental_condition: Contains treatment/control information")
print("• timepoint: Contains temporal sampling information")

# Use HfCacheManager to extract embedded metadata
print("Using HfCacheManager for Metadata Extraction:")
print("=" * 50)

# Extract experimental_condition metadata
success1 = cache_manager._extract_embedded_metadata_field(
    'sample_genomics_data', 
    'experimental_condition', 
    'metadata_experimental_conditions'
)

# Extract timepoint metadata  
success2 = cache_manager._extract_embedded_metadata_field(
    'sample_genomics_data',
    'timepoint', 
    'metadata_timepoints'
)

print(f"Experimental condition extraction: {'✓ Success' if success1 else '✗ Failed'}")
print(f"Timepoint extraction: {'✓ Success' if success2 else '✗ Failed'}")

# Show extracted metadata tables
if success1:
    print(f"\nExtracted experimental conditions:")
    conditions = conn.execute(
        "SELECT value, count FROM metadata_experimental_conditions ORDER BY count DESC"
    ).fetchall()
    
    for condition, count in conditions:
        print(f"  • {condition}: {count} samples")

if success2:
    print(f"\nExtracted timepoints:")
    timepoints = conn.execute(
        "SELECT value, count FROM metadata_timepoints ORDER BY count DESC"
    ).fetchall()
    
    for timepoint, count in timepoints:
        print(f"  • {timepoint}: {count} samples")

print(f"\nBenefits of extraction:")
print("• Separate queryable metadata tables")
print("• Fast metadata-based filtering and analysis") 
print("• Clear separation of data and metadata concerns")
print("• Reusable metadata across different analyses")

In [7]:
from huggingface_hub import scan_cache_dir

# Get current cache information  
cache_info = scan_cache_dir()

print("Current HuggingFace Cache Status:")
print("=" * 35)
print(f"Total size: {cache_info.size_on_disk_str}")
print(f"Number of repositories: {len(cache_info.repos)}")

print("\nRepository breakdown:")
for repo in list(cache_info.repos)[:5]:  # Show first 5 repos
    print(f"  • {repo.repo_id}: {repo.size_on_disk_str} ({len(repo.revisions)} revisions)")

if len(cache_info.repos) > 5:
    print(f"  ... and {len(cache_info.repos) - 5} more repositories")

# Show target repository if it exists in cache
target_repo = None
for repo in cache_info.repos:
    if repo.repo_id == cache_manager.repo_id:
        target_repo = repo
        break

if target_repo:
    print(f"\nTarget repository ({cache_manager.repo_id}) cache info:")
    print(f"  Size: {target_repo.size_on_disk_str}")
    print(f"  Revisions: {len(target_repo.revisions)}")
    if target_repo.revisions:
        latest_rev = max(target_repo.revisions, key=lambda r: r.last_modified)
        print(f"  Latest revision: {latest_rev.commit_hash[:8]}")
        print(f"  Last modified: {latest_rev.last_modified}")
else:
    print(f"\nTarget repository ({cache_manager.repo_id}) not found in cache.")
    print("It may need to be downloaded first.")

Current HuggingFace Cache Status:
Total size: 5.5G
Number of repositories: 11

Repository breakdown:
  • BrentLab/yeast_comparative_analysis: 166.1K (1 revisions)
  • BrentLab/yeast_genome_resources: 114.5K (7 revisions)
  • BrentLab/barkai_compendium: 3.6G (1 revisions)
  • BrentLab/kemmeren_2014: 646.2M (3 revisions)
  • BrentLab/hu_2007_reimand_2010: 42.7M (1 revisions)
  ... and 6 more repositories

Target repository (BrentLab/mahendrawada_2025) cache info:
  Size: 94.3M
  Revisions: 4
  Latest revision: af5ac9dc
  Last modified: 1763578870.280984


### Cache Cleanup by Age

In [8]:
# Clean cache entries older than 30 days (dry run)
print("Cleaning cache by age (30+ days old):")
print("=" * 40)

age_cleanup = cache_manager.clean_cache_by_age(
    max_age_days=30,
    dry_run=True  # Set to False to actually execute
)

print(f"\nCleanup strategy created:")
print(f"Expected space freed: {age_cleanup.expected_freed_size_str}")

# Count total items to delete across all categories
total_items = len(age_cleanup.blobs) + len(age_cleanup.refs) + len(age_cleanup.repos) + len(age_cleanup.snapshots)
print(f"Items to delete: {total_items}")

# Show breakdown of what would be deleted
if total_items > 0:
    print(f"\nBreakdown of items to delete:")
    print(f"  • Blob files: {len(age_cleanup.blobs)}")
    print(f"  • Reference files: {len(age_cleanup.refs)}")
    print(f"  • Repository directories: {len(age_cleanup.repos)}")
    print(f"  • Snapshot directories: {len(age_cleanup.snapshots)}")
    
    # Show some example items
    if age_cleanup.blobs:
        print(f"\nSample blob files to delete:")
        for item in list(age_cleanup.blobs)[:3]:
            print(f"  • {item}")
        if len(age_cleanup.blobs) > 3:
            print(f"  ... and {len(age_cleanup.blobs) - 3} more blob files")
else:
    print("No old files found for cleanup.")

Cleaning cache by age (30+ days old):


INFO:__main__:Found 41 old revisions. Will free 4.7G
INFO:__main__:Dry run completed. Use dry_run=False to execute deletion



Cleanup strategy created:
Expected space freed: 4.7G
Items to delete: 46

Breakdown of items to delete:
  • Blob files: 27
  • Reference files: 0
  • Repository directories: 7
  • Snapshot directories: 12

Sample blob files to delete:
  • /home/chase/.cache/huggingface/hub/datasets--BrentLab--harbison_2004/blobs/b5fbd9e98fd8ddadeeb5631e3b6f5055e917c98d
  • /home/chase/.cache/huggingface/hub/datasets--BrentLab--hackett_2020/blobs/a85bd6b418d9644d9adaa1269c27f97469a4aaee51af63cf1aa041f62cd8ba2c
  • /home/chase/.cache/huggingface/hub/datasets--BrentLab--hackett_2020/blobs/c3e72ccb1b8deba4bbfd18abe6081de7ec3914d9
  ... and 24 more blob files


### Cache Cleanup by Size

In [9]:
# Clean cache to target size (dry run)
target_size = "1GB"
print(f"Cleaning cache to target size: {target_size}")
print("=" * 40)

size_cleanup = cache_manager.clean_cache_by_size(
    target_size=target_size,
    strategy="oldest_first",  # Can be: oldest_first, largest_first, least_used
    dry_run=True
)

print(f"\nSize-based cleanup strategy:")
print(f"Expected space freed: {size_cleanup.expected_freed_size_str}")

# Count total items to delete across all categories
total_items = len(size_cleanup.blobs) + len(size_cleanup.refs) + len(size_cleanup.repos) + len(size_cleanup.snapshots)
print(f"Items to delete: {total_items}")

# Compare different strategies
strategies = ["oldest_first", "largest_first", "least_used"]
print(f"\nComparing cleanup strategies for {target_size}:")

for strategy in strategies:
    try:
        strategy_result = cache_manager.clean_cache_by_size(
            target_size=target_size,
            strategy=strategy,
            dry_run=True
        )
        strategy_total = (len(strategy_result.blobs) + len(strategy_result.refs) + 
                         len(strategy_result.repos) + len(strategy_result.snapshots))
        print(f"  • {strategy:15}: {strategy_result.expected_freed_size_str:>8} "
              f"({strategy_total} items)")
    except Exception as e:
        print(f"  • {strategy:15}: Error - {e}")

Cleaning cache to target size: 1GB


INFO:__main__:Selected 17 revisions for deletion. Will free 3.8G
INFO:__main__:Dry run completed. Use dry_run=False to execute deletion



Size-based cleanup strategy:
Expected space freed: 3.8G
Items to delete: 85

Comparing cleanup strategies for 1GB:


INFO:__main__:Selected 17 revisions for deletion. Will free 3.8G
INFO:__main__:Dry run completed. Use dry_run=False to execute deletion


  • oldest_first   :     3.8G (85 items)


INFO:__main__:Selected 4 revisions for deletion. Will free 4.0G
INFO:__main__:Dry run completed. Use dry_run=False to execute deletion


  • largest_first  :     4.0G (8 items)


INFO:__main__:Selected 17 revisions for deletion. Will free 3.8G
INFO:__main__:Dry run completed. Use dry_run=False to execute deletion


  • least_used     :     3.8G (85 items)


### Cleaning Unused Revisions

In [10]:
# Clean unused revisions, keeping only the latest 2 per repository
print("Cleaning unused revisions (keep latest 2 per repo):")
print("=" * 50)

revision_cleanup = cache_manager.clean_unused_revisions(
    keep_latest=2,
    dry_run=True
)

print(f"\nRevision cleanup strategy:")
print(f"Expected space freed: {revision_cleanup.expected_freed_size_str}")

# Count total items to delete across all categories
total_items = len(revision_cleanup.blobs) + len(revision_cleanup.refs) + len(revision_cleanup.repos) + len(revision_cleanup.snapshots)
print(f"Items to delete: {total_items}")

# Show breakdown
if total_items > 0:
    print(f"\nBreakdown of cleanup:")
    print(f"  • Blob files: {len(revision_cleanup.blobs)}")
    print(f"  • Reference files: {len(revision_cleanup.refs)}")  
    print(f"  • Repository directories: {len(revision_cleanup.repos)}")
    print(f"  • Snapshot directories: {len(revision_cleanup.snapshots)}")

# Show repository-specific breakdown
cache_info = scan_cache_dir()
if cache_info.repos:
    print("\nPer-repository revision analysis:")
    for repo in list(cache_info.repos)[:3]:
        print(f"\n  • {repo.repo_id}:")
        print(f"    Total revisions: {len(repo.revisions)}")
        print(f"    Would keep: {min(2, len(repo.revisions))}")
        print(f"    Would delete: {max(0, len(repo.revisions) - 2)}")
        
        # Show revision details
        sorted_revisions = sorted(repo.revisions, key=lambda r: r.last_modified, reverse=True)
        for i, rev in enumerate(sorted_revisions[:2]):
            print(f"    Keep: {rev.commit_hash[:8]} (modified: {rev.last_modified})")
        
        for rev in sorted_revisions[2:3]:  # Show one that would be deleted
            print(f"    Delete: {rev.commit_hash[:8]} (modified: {rev.last_modified})")

Cleaning unused revisions (keep latest 2 per repo):


INFO:__main__:Found 31 unused revisions. Will free 642.9M
INFO:__main__:Dry run completed. Use dry_run=False to execute deletion



Revision cleanup strategy:
Expected space freed: 642.9M
Items to delete: 118

Breakdown of cleanup:
  • Blob files: 87
  • Reference files: 0
  • Repository directories: 0
  • Snapshot directories: 31

Per-repository revision analysis:

  • BrentLab/yeast_comparative_analysis:
    Total revisions: 1
    Would keep: 1
    Would delete: 0
    Keep: ac03d065 (modified: 1767824941.5531375)

  • BrentLab/yeast_genome_resources:
    Total revisions: 7
    Would keep: 2
    Would delete: 5
    Keep: 42beb284 (modified: 1758155946.5549896)
    Keep: 15fdb72f (modified: 1755819093.2306638)
    Delete: 7441b9a8 (modified: 1755816785.6988702)

  • BrentLab/barkai_compendium:
    Total revisions: 1
    Would keep: 1
    Would delete: 0
    Keep: a987ef37 (modified: 1756926783.3167186)


### Automated Cache Management

In [11]:
# Automated cache cleanup with multiple strategies
print("Automated cache cleanup (comprehensive):")
print("=" * 40)

auto_cleanup = cache_manager.auto_clean_cache(
    max_age_days=30,           # Remove anything older than 30 days
    max_total_size="5GB",      # Target maximum cache size
    keep_latest_per_repo=2,    # Keep 2 latest revisions per repo
    dry_run=True               # Dry run for safety
)

print(f"\nAutomated cleanup executed {len(auto_cleanup)} strategies:")

total_freed = 0
for i, strategy in enumerate(auto_cleanup, 1):
    print(f"  {i}. Strategy freed: {strategy.expected_freed_size_str}")
    total_freed += strategy.expected_freed_size

print(f"\nTotal space that would be freed: {cache_manager._format_bytes(total_freed)}")

# Calculate final cache size
current_cache = scan_cache_dir()
final_size = current_cache.size_on_disk - total_freed
print(f"Cache size after cleanup: {cache_manager._format_bytes(max(0, final_size))}")

INFO:__main__:Starting automated cache cleanup...


Automated cache cleanup (comprehensive):


INFO:__main__:Found 41 old revisions. Will free 4.7G
INFO:__main__:Dry run completed. Use dry_run=False to execute deletion
INFO:__main__:Found 31 unused revisions. Will free 642.9M
INFO:__main__:Dry run completed. Use dry_run=False to execute deletion
INFO:__main__:Selected 9 revisions for deletion. Will free 2.8M
INFO:__main__:Dry run completed. Use dry_run=False to execute deletion
INFO:__main__:Automated cleanup complete. Total freed: 5.0GB



Automated cleanup executed 3 strategies:
  1. Strategy freed: 4.7G
  2. Strategy freed: 642.9M
  3. Strategy freed: 2.8M

Total space that would be freed: 5.0GB
Cache size after cleanup: 129.8MB


## 7. Best Practices and Performance Tips

Here are some best practices for using HfCacheManager effectively:

### Performance Best Practices

In [12]:
import time

print("Performance Demonstration: Cache Management Benefits")
print("=" * 55)

print("\nDemonstrating cache cleanup performance...")

# Show performance of cache scanning and cleanup strategy creation
print("\n1. Cache scanning performance:")
start_time = time.time()
cache_info = scan_cache_dir()
scan_time = time.time() - start_time
print(f"   Time to scan cache: {scan_time:.3f} seconds")
print(f"   Repositories found: {len(cache_info.repos)}")
print(f"   Total cache size: {cache_info.size_on_disk_str}")

# Show performance of cleanup strategy creation
print("\n2. Cleanup strategy creation performance:")

start_time = time.time()
age_strategy = cache_manager.clean_cache_by_age(max_age_days=30, dry_run=True)
age_time = time.time() - start_time
print(f"   Age cleanup strategy: {age_time:.3f} seconds")

start_time = time.time()
size_strategy = cache_manager.clean_cache_by_size(target_size="1GB", dry_run=True)
size_time = time.time() - start_time
print(f"   Size cleanup strategy: {size_time:.3f} seconds")

start_time = time.time()
revision_strategy = cache_manager.clean_unused_revisions(keep_latest=2, dry_run=True)
revision_time = time.time() - start_time
print(f"   Revision cleanup strategy: {revision_time:.3f} seconds")

print(f"\nPerformance insights:")
print(f"• Cache scanning is fast: {scan_time:.3f}s for {len(cache_info.repos)} repos")
print(f"• Cleanup strategy creation is efficient")
print(f"• Dry runs allow safe preview of cleanup operations")
print(f"• Multiple strategies can be compared quickly")

Performance Demonstration: Cache Management Benefits

Demonstrating cache cleanup performance...

1. Cache scanning performance:
   Time to scan cache: 0.096 seconds
   Repositories found: 11
   Total cache size: 5.5G

2. Cleanup strategy creation performance:


INFO:__main__:Found 41 old revisions. Will free 4.7G
INFO:__main__:Dry run completed. Use dry_run=False to execute deletion


   Age cleanup strategy: 0.094 seconds


INFO:__main__:Selected 17 revisions for deletion. Will free 3.8G
INFO:__main__:Dry run completed. Use dry_run=False to execute deletion
INFO:__main__:Found 31 unused revisions. Will free 642.9M
INFO:__main__:Dry run completed. Use dry_run=False to execute deletion


   Size cleanup strategy: 0.093 seconds
   Revision cleanup strategy: 0.100 seconds

Performance insights:
• Cache scanning is fast: 0.096s for 11 repos
• Cleanup strategy creation is efficient
• Dry runs allow safe preview of cleanup operations
• Multiple strategies can be compared quickly


### Memory and Storage Optimization

In [13]:
print("Memory and Storage Optimization Tips:")
print("=" * 40)

print("\n1. DuckDB Views vs Tables:")
print("   • HfCacheManager creates VIEWS by default (not tables)")
print("   • Views reference original parquet files without duplication")
print("   • This saves storage space while enabling fast SQL queries")

print("\n2. Metadata-First Workflow:")
print("   • Load metadata first to understand data structure")
print("   • Use metadata to filter and select specific data subsets")
print("   • Avoid loading entire datasets when only portions are needed")

print("\n3. Cache Management Strategy:")
print("   • Run automated cleanup regularly")
print("   • Keep cache size reasonable for your system")
print("   • Prioritize keeping recent and frequently-used datasets")

# Demonstrate DuckDB view benefits
tables_info = conn.execute(
    "SELECT table_name, table_type FROM information_schema.tables WHERE table_name LIKE 'metadata_%'"
).fetchall()

if tables_info:
    print(f"\nCurrent DuckDB objects ({len(tables_info)} total):")
    for table_name, table_type in tables_info:
        print(f"   • {table_name}: {table_type}")
    
    view_count = sum(1 for _, table_type in tables_info if table_type == 'VIEW')
    print(f"\n   {view_count} views created (space-efficient!)")

Memory and Storage Optimization Tips:

1. DuckDB Views vs Tables:
   • HfCacheManager creates VIEWS by default (not tables)
   • Views reference original parquet files without duplication
   • This saves storage space while enabling fast SQL queries

2. Metadata-First Workflow:
   • Load metadata first to understand data structure
   • Use metadata to filter and select specific data subsets
   • Avoid loading entire datasets when only portions are needed

3. Cache Management Strategy:
   • Run automated cleanup regularly
   • Keep cache size reasonable for your system
   • Prioritize keeping recent and frequently-used datasets


## 8. Integration with Other Components

The HfCacheManager works seamlessly with other components in the labretriever ecosystem.

In [14]:
print("HfCacheManager Integration Workflow:")
print("=" * 40)

print("\n1. Cache Management Setup:")
print("   from labretriever.HfCacheManager import HfCacheManager")
print("   cache_mgr = HfCacheManager(repo_id, duckdb_conn)")
print("   # Inherits all DataCard functionality + cache management")

print("\n2. Proactive Cache Cleanup:")
print("   # Clean before large operations")
print("   cache_mgr.auto_clean_cache(max_total_size='5GB', dry_run=False)")
print("   # Or use specific strategies")
print("   cache_mgr.clean_cache_by_age(max_age_days=30)")

print("\n3. Data Loading with Cache Awareness:")
print("   # The 3-case strategy works automatically with HfQueryAPI")
print("   from labretriever import HfQueryAPI")
print("   query_api = HfQueryAPI(repo_id, duckdb_conn)")
print("   # Metadata loading uses cache manager's strategy")
print("   data_df = query_api.get_pandas('config_name')")

print("\n4. Embedded Metadata Extraction:")
print("   # Extract metadata fields after data loading")
print("   cache_mgr._extract_embedded_metadata_field(")
print("       'data_table_name', 'metadata_field', 'metadata_table_name')")

print("\n5. Regular Cache Maintenance:")
print("   # Schedule regular cleanup")
print("   cache_mgr.clean_unused_revisions(keep_latest=2)")
print("   cache_mgr.clean_cache_by_size('10GB', strategy='oldest_first')")

# Show current state
print(f"\nCurrent Session State:")
print(f"Repository: {cache_manager.repo_id}")
print(f"DuckDB tables: {len(conn.execute('SELECT table_name FROM information_schema.tables').fetchall())}")

cache_info = scan_cache_dir()
print(f"HF cache size: {cache_info.size_on_disk_str}")
print(f"Cache repositories: {len(cache_info.repos)}")

HfCacheManager Integration Workflow:

1. Cache Management Setup:
   from labretriever.HfCacheManager import HfCacheManager
   cache_mgr = HfCacheManager(repo_id, duckdb_conn)
   # Inherits all DataCard functionality + cache management

2. Proactive Cache Cleanup:
   # Clean before large operations
   cache_mgr.auto_clean_cache(max_total_size='5GB', dry_run=False)
   # Or use specific strategies
   cache_mgr.clean_cache_by_age(max_age_days=30)

3. Data Loading with Cache Awareness:
   # The 3-case strategy works automatically with HfQueryAPI
   from labretriever import HfQueryAPI
   query_api = HfQueryAPI(repo_id, duckdb_conn)
   # Metadata loading uses cache manager's strategy
   data_df = query_api.get_pandas('config_name')

4. Embedded Metadata Extraction:
   # Extract metadata fields after data loading
   cache_mgr._extract_embedded_metadata_field(
       'data_table_name', 'metadata_field', 'metadata_table_name')

5. Regular Cache Maintenance:
   # Schedule regular cleanup
   cache

## 9. Troubleshooting and Error Handling

The HfCacheManager includes comprehensive error handling and diagnostic capabilities.

In [15]:
print("Cache Management Troubleshooting:")
print("=" * 35)

print("\n1. Import and Setup Issues:")
print("   • Ensure correct import: from labretriever.HfCacheManager import HfCacheManager")
print("   • Verify DuckDB connection: conn = duckdb.connect(':memory:')")
print("   • Check repository access permissions")

print("\n2. Cache Space and Performance Issues:")
try:
    cache_info = scan_cache_dir()
    print(f"   Current cache size: {cache_info.size_on_disk_str}")
    print("   • Use auto_clean_cache() for automated management")
    print("   • Monitor cache growth with scan_cache_dir()")
    print("   • Set appropriate size limits for your system")
    
    # Show if cache is getting large
    total_gb = cache_info.size_on_disk / (1024**3)
    if total_gb > 10:
        print(f"   ⚠️  Large cache detected ({total_gb:.1f}GB) - consider cleanup")
    
except Exception as e:
    print(f"   Cache scan error: {e}")

print("\n3. Cache Cleanup Issues:")
print("   • Use dry_run=True first to preview changes")
print("   • Check disk permissions for cache directory")
print("   • Verify no active processes are using cached files")

print("\n4. DuckDB Integration Issues:")
print("   • Ensure DuckDB connection is active")
print("   • Check memory limits for in-memory databases")
print("   • Verify table names don't conflict")

# Perform health checks
print(f"\nCache Health Check:")

# Test DuckDB
try:
    test_result = conn.execute("SELECT 'DuckDB OK' as status").fetchone()
    print(f"✓ DuckDB connection: {test_result[0]}")
except Exception as e:
    print(f"✗ DuckDB connection: {e}")

# Test cache access
try:
    cache_info = scan_cache_dir()
    print(f"✓ Cache access: {len(cache_info.repos)} repositories found")
except Exception as e:
    print(f"✗ Cache access: {e}")

# Test cache manager methods
try:
    test_cleanup = cache_manager.clean_cache_by_age(max_age_days=999, dry_run=True)
    print(f"✓ Cache cleanup methods: Working")
except Exception as e:
    print(f"✗ Cache cleanup methods: {e}")

print(f"\nCurrent Status:")
print(f"Repository: {cache_manager.repo_id}")
print(f"Logger configured: {cache_manager.logger is not None}")
print(f"Cache management ready: ✓")

Cache Management Troubleshooting:

1. Import and Setup Issues:
   • Ensure correct import: from labretriever.HfCacheManager import HfCacheManager
   • Verify DuckDB connection: conn = duckdb.connect(':memory:')
   • Check repository access permissions

2. Cache Space and Performance Issues:
   Current cache size: 5.5G
   • Use auto_clean_cache() for automated management
   • Monitor cache growth with scan_cache_dir()
   • Set appropriate size limits for your system

3. Cache Cleanup Issues:
   • Use dry_run=True first to preview changes
   • Check disk permissions for cache directory
   • Verify no active processes are using cached files

4. DuckDB Integration Issues:
   • Ensure DuckDB connection is active
   • Check memory limits for in-memory databases
   • Verify table names don't conflict

Cache Health Check:
✓ DuckDB connection: DuckDB OK
✓ Cache access: 11 repositories found


INFO:__main__:No old revisions found to delete
INFO:__main__:Found 0 old revisions. Will free 0.0
INFO:__main__:Dry run completed. Use dry_run=False to execute deletion


✓ Cache cleanup methods: Working

Current Status:
Repository: BrentLab/mahendrawada_2025
Logger configured: True
Cache management ready: ✓
